# Advanced Problems with Solutions: Yielding and Generators

This notebook contains advanced, testable practice problems on Python generators, `yield`, iterator exhaustion, `yield from`, lazy pipelines, infinite streams, generator delegation, and coroutine-style `.send()` usage.

Each problem follows this pattern:

1. **Problem statement**
2. **Best-practice notes**
3. **Solution code**
4. **Tests / examples**

> Run the notebook from top to bottom. All `assert` statements should pass.

## Setup: imports and small testing helpers

In [1]:
from __future__ import annotations

from collections import Counter, deque
from collections.abc import Iterable as ABCIterable
from dataclasses import dataclass
from itertools import islice
from typing import Any, Callable, Iterable, Iterator, TypeVar
import heapq
import inspect
import math
import re

T = TypeVar("T")
U = TypeVar("U")


def take(n: int, iterable: Iterable[T]) -> list[T]:
    """Return the first n items from any iterable without consuming more."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return list(islice(iterable, n))


def assert_raises(expected_exception: type[BaseException], func: Callable[..., Any], *args: Any, **kwargs: Any) -> BaseException:
    """Small assertion helper for exception-based tests."""
    try:
        func(*args, **kwargs)
    except expected_exception as exc:
        return exc
    raise AssertionError(f"Expected {expected_exception.__name__} to be raised")

## Warm-up example: generator states

A function containing `yield` does not run immediately when called. It returns a generator object. The body runs only when the generator is advanced with `next()`, a `for` loop, `list()`, `islice()`, etc.

In [2]:
def tiny_story() -> Iterator[str]:
    yield "created lazily"
    yield "resumed later"
    return "finished"


g = tiny_story()

states = [inspect.getgeneratorstate(g)]
first = next(g)
states.append(inspect.getgeneratorstate(g))
second = next(g)
states.append(inspect.getgeneratorstate(g))

try:
    next(g)
except StopIteration as exc:
    final_return_value = exc.value

states.append(inspect.getgeneratorstate(g))

assert first == "created lazily"
assert second == "resumed later"
assert final_return_value == "finished"
assert states == ["GEN_CREATED", "GEN_SUSPENDED", "GEN_SUSPENDED", "GEN_CLOSED"]

states

['GEN_CREATED', 'GEN_SUSPENDED', 'GEN_SUSPENDED', 'GEN_CLOSED']

---

# Problem 1 — Rewrite a class-based iterator as a generator

## Problem

Write a generator function `factorials(n)` that yields:

```python
0!, 1!, 2!, ..., (n-1)!
```

For example:

```python
list(factorials(6)) == [1, 1, 2, 6, 24, 120]
```

## Requirements

- Use `yield`.
- Do not call `math.factorial` inside the loop.
- Validate that `n` is non-negative.
- Demonstrate that the returned generator is **one-shot**: once exhausted, it stays exhausted.

## Best-practice notes

- Prefer a generator function when the iterator state is simple.
- Avoid recomputing expensive values when you can update state incrementally.
- Treat generator objects as one-time streams, not reusable containers.

In [3]:
def factorials(n: int) -> Iterator[int]:
    """Yield 0!, 1!, ..., (n-1)! lazily."""
    if n < 0:
        raise ValueError("n must be non-negative")

    result = 1
    for i in range(n):
        if i > 0:
            result *= i
        yield result


assert list(factorials(0)) == []
assert list(factorials(1)) == [1]
assert list(factorials(6)) == [1, 1, 2, 6, 24, 120]

assert_raises(ValueError, lambda: list(factorials(-1)))

facts = factorials(4)
assert list(facts) == [1, 1, 2, 6]
assert list(facts) == []  # exhausted generators do not restart

list(factorials(8))

[1, 1, 2, 6, 24, 120, 720, 5040]

---

# Problem 2 — Prove that a generator pipeline is lazy

## Problem

Create a lazy pipeline that:

1. receives numbers,
2. keeps only even numbers,
3. squares them,
4. returns the first three results.

Then prove that the source iterable was not fully consumed.

## Requirements

- Do not build intermediate lists.
- Use a custom iterable that counts how many source items were actually produced.
- The first three even squares from `range(...)` should be `[0, 4, 16]`.
- The source should only produce five values: `0, 1, 2, 3, 4`.

In [4]:
class CountingRange:
    """Iterable that records how many items have been pulled from it."""
    def __init__(self, stop: int) -> None:
        self.stop = stop
        self.produced = 0

    def __iter__(self) -> Iterator[int]:
        for value in range(self.stop):
            self.produced += 1
            yield value


def even_squares(numbers: Iterable[int]) -> Iterator[int]:
    """Yield squares of even numbers lazily."""
    for number in numbers:
        if number % 2 == 0:
            yield number * number


source = CountingRange(1_000_000)
first_three = take(3, even_squares(source))

assert first_three == [0, 4, 16]
assert source.produced == 5

first_three, source.produced

([0, 4, 16], 5)

---

# Problem 3 — Sliding windows and chunks

## Problem

Implement two reusable generator utilities:

1. `sliding_window(iterable, size, step=1)`
2. `chunked(iterable, size)`

## Requirements

For `sliding_window`:

```python
list(sliding_window([10, 20, 30, 40], 3))
# [(10, 20, 30), (20, 30, 40)]
```

For `chunked`:

```python
list(chunked(range(7), 3))
# [(0, 1, 2), (3, 4, 5), (6,)]
```

## Best-practice notes

- Accept any iterable, not only lists.
- Do not index the input.
- Validate arguments early.
- Use tuples for yielded groups so downstream code cannot mutate your internal buffer.

In [5]:
def sliding_window(iterable: Iterable[T], size: int, step: int = 1) -> Iterator[tuple[T, ...]]:
    """Yield fixed-size sliding windows from any iterable."""
    if size <= 0:
        raise ValueError("size must be positive")
    if step <= 0:
        raise ValueError("step must be positive")

    iterator = iter(iterable)
    window: deque[T] = deque(maxlen=size)

    for _ in range(size):
        try:
            window.append(next(iterator))
        except StopIteration:
            return

    yield tuple(window)

    while True:
        try:
            for _ in range(step):
                window.append(next(iterator))
        except StopIteration:
            return
        yield tuple(window)


def chunked(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    """Yield tuples of at most size items from any iterable."""
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    while True:
        chunk = tuple(islice(iterator, size))
        if not chunk:
            return
        yield chunk


assert list(sliding_window([10, 20, 30, 40], 3)) == [(10, 20, 30), (20, 30, 40)]
assert list(sliding_window(range(7), 3, step=2)) == [(0, 1, 2), (2, 3, 4), (4, 5, 6)]
assert list(sliding_window([1, 2], 3)) == []

assert list(chunked(range(7), 3)) == [(0, 1, 2), (3, 4, 5), (6,)]
assert list(chunked([], 3)) == []

assert_raises(ValueError, lambda: list(sliding_window([1, 2, 3], 0)))
assert_raises(ValueError, lambda: list(chunked([1, 2, 3], 0)))

list(sliding_window("ABCDEFG", 4, step=2))

[('A', 'B', 'C', 'D'), ('C', 'D', 'E', 'F')]

---

# Problem 4 — Recursive flattening with `yield from`

## Problem

Write `flatten(items)` that recursively yields values from nested iterables.

## Requirements

- Use `yield from`.
- Treat strings and bytes as atomic values, not as iterables to be expanded.
- Work with lists, tuples, ranges, sets, and generators.
- Avoid returning a list from `flatten`; it must be lazy.

In [6]:
def flatten(items: Iterable[Any]) -> Iterator[Any]:
    """Recursively flatten nested iterables, treating strings/bytes as atomic."""
    for item in items:
        if isinstance(item, (str, bytes)):
            yield item
        elif isinstance(item, ABCIterable):
            yield from flatten(item)
        else:
            yield item


nested = [1, [2, (3, 4)], "abc", [range(5, 8), (x for x in [8, 9])], b"xy"]
assert list(flatten(nested)) == [1, 2, 3, 4, "abc", 5, 6, 7, 8, 9, b"xy"]

deep = [[[[1]]], 2, [[[3, [4]]]]]
assert list(flatten(deep)) == [1, 2, 3, 4]

list(flatten(nested))

[1, 2, 3, 4, 'abc', 5, 6, 7, b'xy']

---

# Problem 5 — Build a re-iterable wrapper around a generator factory

## Problem

A generator object is an iterator and is consumed after one pass. Sometimes you need a reusable iterable that creates a **fresh generator** every time iteration starts.

Implement `FreshIterable`.

## Requirements

- `FreshIterable(factory)` accepts a zero-argument function that returns an iterable.
- `iter(FreshIterable(...))` must return a fresh iterator each time.
- Demonstrate the difference between a one-shot generator object and a reusable generator factory.

In [7]:
class FreshIterable:
    """Reusable iterable backed by a factory that creates a fresh iterable."""
    def __init__(self, factory: Callable[[], Iterable[T]]) -> None:
        self._factory = factory

    def __iter__(self) -> Iterator[T]:
        return iter(self._factory())


def evens_less_than(limit: int) -> Iterator[int]:
    for value in range(limit):
        if value % 2 == 0:
            yield value


one_shot = evens_less_than(7)
assert list(one_shot) == [0, 2, 4, 6]
assert list(one_shot) == []

reusable = FreshIterable(lambda: evens_less_than(7))
assert list(reusable) == [0, 2, 4, 6]
assert list(reusable) == [0, 2, 4, 6]

# FreshIterable also works with non-generator iterables.
letters = FreshIterable(lambda: "ABC")
assert list(letters) == ["A", "B", "C"]
assert list(letters) == ["A", "B", "C"]

list(reusable)

[0, 2, 4, 6]

---

# Problem 6 — Capture a subgenerator's return value with `yield from`

## Problem

A generator can `return value`. That value becomes the `.value` attribute of the `StopIteration` exception. When using `yield from`, the delegating generator can capture that returned value.

Implement:

- `count_and_sum(n)` which yields numbers from `1` to `n` and returns their sum.
- `report_sum(n)` which delegates with `yield from`, captures the returned sum, then yields a final report string.

## Requirements

```python
list(report_sum(4)) == [1, 2, 3, 4, "sum=10"]
```

In [8]:
def count_and_sum(n: int) -> Iterator[int]:
    """Yield 1..n, then return their sum as StopIteration.value."""
    if n < 0:
        raise ValueError("n must be non-negative")

    total = 0
    for value in range(1, n + 1):
        total += value
        yield value
    return total


def report_sum(n: int) -> Iterator[int | str]:
    """Delegate to count_and_sum, then yield a summary message."""
    total = yield from count_and_sum(n)
    yield f"sum={total}"


assert list(report_sum(4)) == [1, 2, 3, 4, "sum=10"]
assert list(report_sum(0)) == ["sum=0"]

manual = count_and_sum(3)
seen = []

while True:
    try:
        seen.append(next(manual))
    except StopIteration as exc:
        returned = exc.value
        break

assert seen == [1, 2, 3]
assert returned == 6

list(report_sum(6))

[1, 2, 3, 4, 5, 6, 'sum=21']

---

# Problem 7 — Use `.send()` to build a running average generator

## Problem

Create a coroutine-style generator `running_average()`.

## Behavior

- Prime it with `next(avg)`.
- Send numbers with `avg.send(number)`.
- Each send returns the current running average.
- Sending `None` should not change the average.
- The generator should keep its state internally.

## Best-practice notes

- A generator must be advanced to its first `yield` before sending a non-`None` value.
- Keep coroutine-style protocols small and well documented.
- Use `.close()` when you are finished with a long-lived generator.

In [9]:
def running_average() -> Iterator[float | None]:
    """Receive numbers through .send() and yield the current average."""
    total = 0.0
    count = 0
    average: float | None = None

    while True:
        value = yield average
        if value is None:
            continue
        total += float(value)
        count += 1
        average = total / count


avg = running_average()

# Generator is created but not started yet.
assert inspect.getgeneratorstate(avg) == "GEN_CREATED"

assert next(avg) is None
assert inspect.getgeneratorstate(avg) == "GEN_SUSPENDED"

assert avg.send(10) == 10.0
assert avg.send(20) == 15.0
assert avg.send(30) == 20.0
assert avg.send(None) == 20.0

avg.close()
assert inspect.getgeneratorstate(avg) == "GEN_CLOSED"

---

# Problem 8 — Stream and validate CSV-like rows

## Problem

Build a streaming parser for simple CSV-like text.

## Requirements

Implement two generators:

1. `rows_from_csv_lines(lines)`
   - Reads a header from the first non-empty, non-comment line.
   - Skips blank lines and comment lines beginning with `#`.
   - Yields dictionaries for valid rows.
   - Yields error dictionaries for malformed rows.

2. `validate_people(rows)`
   - Converts `age` to `int`.
   - Emits an error dictionary for invalid or negative ages.
   - Otherwise yields cleaned records.

## Best-practice notes

- Keep parsing and validation as separate pipeline stages.
- Yield structured errors instead of crashing when processing streams.
- Preserve line numbers for debuggability.

In [10]:
def meaningful_lines(lines: Iterable[str]) -> Iterator[tuple[int, str]]:
    """Yield (line_number, stripped_line) for non-empty, non-comment lines."""
    for line_number, raw in enumerate(lines, start=1):
        stripped = raw.strip()
        if not stripped or stripped.startswith("#"):
            continue
        yield line_number, stripped


def rows_from_csv_lines(lines: Iterable[str], delimiter: str = ",") -> Iterator[dict[str, Any]]:
    """Parse simple delimited rows into dictionaries."""
    line_iter = meaningful_lines(lines)

    try:
        header_line_number, header_line = next(line_iter)
    except StopIteration:
        return

    headers = [part.strip() for part in header_line.split(delimiter)]

    for line_number, line in line_iter:
        parts = [part.strip() for part in line.split(delimiter)]

        if len(parts) != len(headers):
            yield {
                "_error": "wrong_number_of_fields",
                "line_number": line_number,
                "expected": len(headers),
                "actual": len(parts),
                "raw": line,
            }
            continue

        row = dict(zip(headers, parts))
        row["_line_number"] = line_number
        yield row


def validate_people(rows: Iterable[dict[str, Any]]) -> Iterator[dict[str, Any]]:
    """Validate and clean people rows emitted by rows_from_csv_lines."""
    for row in rows:
        if "_error" in row:
            yield row
            continue

        try:
            age = int(row["age"])
        except (KeyError, ValueError):
            yield {"_error": "invalid_age", **row}
            continue

        if age < 0:
            yield {"_error": "negative_age", **row}
            continue

        yield {**row, "age": age}


csv_text = """
# demo input
name, age, city
Alice, 30, Sofia
Bob, not_an_int, Paris
Cara, -2, Berlin
Dana, 41
Eve, 22, Rome
"""

records = list(validate_people(rows_from_csv_lines(csv_text.splitlines())))

assert records[0]["name"] == "Alice"
assert records[0]["age"] == 30

assert records[1]["_error"] == "invalid_age"
assert records[2]["_error"] == "negative_age"
assert records[3]["_error"] == "wrong_number_of_fields"

assert records[4] == {"name": "Eve", "age": 22, "city": "Rome", "_line_number": 8}

records

[{'name': 'Alice', 'age': 30, 'city': 'Sofia', '_line_number': 4},
 {'_error': 'invalid_age',
  'name': 'Bob',
  'age': 'not_an_int',
  'city': 'Paris',
  '_line_number': 5},
 {'_error': 'negative_age',
  'name': 'Cara',
  'age': '-2',
  'city': 'Berlin',
  '_line_number': 6},
 {'_error': 'wrong_number_of_fields',
  'line_number': 7,
  'expected': 3,
  'actual': 2,
  'raw': 'Dana, 41'},
 {'name': 'Eve', 'age': 22, 'city': 'Rome', '_line_number': 8}]

---

# Problem 9 — Infinite prime-number stream

## Problem

Write an infinite generator `primes()` that yields prime numbers forever.

## Requirements

- The generator must be infinite.
- Use `itertools.islice` or the provided `take()` helper to safely consume a finite prefix.
- Do not call `list(primes())`.
- Use previously found primes to test new candidates.

In [11]:
def primes() -> Iterator[int]:
    """Yield prime numbers forever."""
    yield 2

    found = [2]
    candidate = 3

    while True:
        limit = math.isqrt(candidate)
        is_prime = True

        for prime in found:
            if prime > limit:
                break
            if candidate % prime == 0:
                is_prime = False
                break

        if is_prime:
            found.append(candidate)
            yield candidate

        candidate += 2


assert take(10, primes()) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

first_100 = take(100, primes())
assert len(first_100) == 100
assert first_100[-1] == 541

take(15, primes())

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

---

# Problem 10 — Stateful event-stream sessionization

## Problem

You receive a time-ordered stream of events. Create user sessions lazily.

A session ends when a user has been idle for more than `idle_timeout` seconds. The stream may contain events from multiple users.

## Requirements

- Define an `Event` dataclass.
- Implement `sessionize_by_user(events, idle_timeout)`.
- Yield session dictionaries with:
  - `user`
  - `start`
  - `end`
  - `actions`
- Emit expired sessions as soon as a later event proves they are expired.
- At the end, flush remaining active sessions.

In [12]:
@dataclass(frozen=True)
class Event:
    user: str
    timestamp: int
    action: str


def new_session(event: Event) -> dict[str, Any]:
    return {
        "user": event.user,
        "start": event.timestamp,
        "end": event.timestamp,
        "actions": [event.action],
    }


def add_to_session(session: dict[str, Any], event: Event) -> None:
    session["end"] = event.timestamp
    session["actions"].append(event.action)


def sessionize_by_user(events: Iterable[Event], idle_timeout: int) -> Iterator[dict[str, Any]]:
    """Yield completed sessions from a time-ordered event stream."""
    if idle_timeout < 0:
        raise ValueError("idle_timeout must be non-negative")

    active: dict[str, dict[str, Any]] = {}

    for event in events:
        expired_users = [
            user
            for user, session in active.items()
            if event.timestamp - session["end"] > idle_timeout
        ]

        for user in sorted(expired_users, key=lambda u: active[u]["end"]):
            yield active.pop(user)

        session = active.get(event.user)
        if session is None:
            active[event.user] = new_session(event)
        else:
            add_to_session(session, event)

    for user in sorted(active, key=lambda u: active[u]["end"]):
        yield active[user]


events = [
    Event("alice", 0, "view"),
    Event("bob", 1, "view"),
    Event("alice", 5, "click"),
    Event("bob", 50, "buy"),
    Event("alice", 100, "logout"),
]

sessions = list(sessionize_by_user(events, idle_timeout=10))

assert sessions == [
    {"user": "bob", "start": 1, "end": 1, "actions": ["view"]},
    {"user": "alice", "start": 0, "end": 5, "actions": ["view", "click"]},
    {"user": "bob", "start": 50, "end": 50, "actions": ["buy"]},
    {"user": "alice", "start": 100, "end": 100, "actions": ["logout"]},
]

sessions

[{'user': 'bob', 'start': 1, 'end': 1, 'actions': ['view']},
 {'user': 'alice', 'start': 0, 'end': 5, 'actions': ['view', 'click']},
 {'user': 'bob', 'start': 50, 'end': 50, 'actions': ['buy']},
 {'user': 'alice', 'start': 100, 'end': 100, 'actions': ['logout']}]

---

# Problem 11 — Lazily merge sorted streams

## Problem

Implement `merge_sorted(*iterables, key=lambda x: x)`.

It should merge multiple already-sorted input streams into one sorted output stream.

## Requirements

- Do not materialize all input items.
- Work with any iterable, including generators.
- Support a custom key function.
- Keep only one pending item from each input stream in memory.

In [13]:
def merge_sorted(*iterables: Iterable[T], key: Callable[[T], Any] = lambda x: x) -> Iterator[T]:
    """Lazily merge sorted iterables."""
    iterators = [iter(iterable) for iterable in iterables]
    heap: list[tuple[Any, int, T]] = []

    for index, iterator in enumerate(iterators):
        try:
            item = next(iterator)
        except StopIteration:
            continue
        heapq.heappush(heap, (key(item), index, item))

    while heap:
        _, index, item = heapq.heappop(heap)
        yield item

        try:
            next_item = next(iterators[index])
        except StopIteration:
            continue

        heapq.heappush(heap, (key(next_item), index, next_item))


assert list(merge_sorted([1, 4, 10], [2, 3, 9], [-1, 12])) == [-1, 1, 2, 3, 4, 9, 10, 12]

people_a = [{"name": "Ada", "age": 20}, {"name": "Grace", "age": 50}]
people_b = [{"name": "Linus", "age": 30}, {"name": "Guido", "age": 60}]

merged_people = list(merge_sorted(people_a, people_b, key=lambda person: person["age"]))
assert [person["name"] for person in merged_people] == ["Ada", "Linus", "Grace", "Guido"]

list(merge_sorted([1, 5, 9], [2, 4, 8], [0, 3, 7]))

[0, 1, 2, 3, 4, 5, 7, 8, 9]

---

# Problem 12 — Capstone: streaming log analytics pipeline

## Problem

Build a log analytics pipeline using small generator stages.

Each log line has this shape:

```text
2026-01-01T00:00:01 INFO user=alice path=/home ms=34
```

## Requirements

Implement generator stages:

1. `parse_logs(lines)`
2. `only_valid(rows)`
3. `only_slow(rows, threshold_ms)`
4. `running_counts(rows, field)`

The final pipeline should parse logs, keep valid logs, filter slow requests, and attach a running per-user count.

## Best-practice notes

- Make every stage do one thing.
- Do not hide parsing errors; yield structured errors first, then choose whether to filter them.
- Keep each stage lazy and composable.

In [14]:
LOG_RE = re.compile(
    r"^(?P<ts>\S+)\s+"
    r"(?P<level>INFO|WARN|ERROR)\s+"
    r"user=(?P<user>\w+)\s+"
    r"path=(?P<path>\S+)\s+"
    r"ms=(?P<ms>\d+)$"
)


def parse_logs(lines: Iterable[str]) -> Iterator[dict[str, Any]]:
    """Parse log lines into dictionaries; yield structured errors for bad lines."""
    for line_number, raw in enumerate(lines, start=1):
        line = raw.strip()
        if not line:
            continue

        match = LOG_RE.match(line)
        if match is None:
            yield {
                "_error": "parse_failed",
                "line_number": line_number,
                "raw": line,
            }
            continue

        row = match.groupdict()
        row["ms"] = int(row["ms"])
        row["line_number"] = line_number
        yield row


def only_valid(rows: Iterable[dict[str, Any]]) -> Iterator[dict[str, Any]]:
    """Drop structured error rows."""
    for row in rows:
        if "_error" not in row:
            yield row


def only_slow(rows: Iterable[dict[str, Any]], threshold_ms: int) -> Iterator[dict[str, Any]]:
    """Yield rows whose ms value is at least threshold_ms."""
    for row in rows:
        if row["ms"] >= threshold_ms:
            yield row


def running_counts(rows: Iterable[dict[str, Any]], field: str) -> Iterator[dict[str, Any]]:
    """Attach a running count per field value."""
    counts: Counter[Any] = Counter()

    for row in rows:
        value = row[field]
        counts[value] += 1
        yield {**row, f"{field}_running_count": counts[value]}


raw_logs = [
    "2026-01-01T00:00:01 INFO user=alice path=/home ms=34",
    "2026-01-01T00:00:02 WARN user=bob path=/cart ms=240",
    "bad line that should not crash the pipeline",
    "2026-01-01T00:00:03 INFO user=alice path=/search ms=150",
    "2026-01-01T00:00:04 ERROR user=bob path=/pay ms=900",
    "2026-01-01T00:00:05 INFO user=cara path=/home ms=50",
]

all_parsed = list(parse_logs(raw_logs))
assert all_parsed[2]["_error"] == "parse_failed"

pipeline = running_counts(
    only_slow(
        only_valid(
            parse_logs(raw_logs)
        ),
        threshold_ms=100,
    ),
    field="user",
)

slow_rows = list(pipeline)

assert [row["user"] for row in slow_rows] == ["bob", "alice", "bob"]
assert [row["user_running_count"] for row in slow_rows] == [1, 1, 2]
assert [row["ms"] for row in slow_rows] == [240, 150, 900]

slow_rows

[{'ts': '2026-01-01T00:00:02',
  'level': 'WARN',
  'user': 'bob',
  'path': '/cart',
  'ms': 240,
  'line_number': 2,
  'user_running_count': 1},
 {'ts': '2026-01-01T00:00:03',
  'level': 'INFO',
  'user': 'alice',
  'path': '/search',
  'ms': 150,
  'line_number': 4,
  'user_running_count': 1},
 {'ts': '2026-01-01T00:00:04',
  'level': 'ERROR',
  'user': 'bob',
  'path': '/pay',
  'ms': 900,
  'line_number': 5,
  'user_running_count': 2}]

---

# Extra practice prompts

Use these as additional challenges after completing the solved problems.

1. Modify `chunked()` to support `strict=True`, raising an error when the final chunk is incomplete.
2. Add a `max_depth` argument to `flatten()`.
3. Add `reset` support to `running_average()` by sending the string `"RESET"`.
4. Modify `rows_from_csv_lines()` to support quoted commas using the standard library `csv` module.
5. Extend `sessionize_by_user()` to reject out-of-order events.
6. Implement a memory-safe `unique_everseen(iterable, key=None)` generator.
7. Implement `round_robin(*iterables)` that alternates between active iterables until all are exhausted.
8. Implement `peekable(iterable)` with `.peek()` and normal iteration behavior.
9. Implement `tail(iterable, n)` using a bounded `deque`.
10. Write property-style tests for `merge_sorted()` using randomly generated sorted lists.

## Best-practices checklist

- Use generator functions when state is simple and output can be streamed.
- Do not call `list()` on unbounded or very large generators.
- Keep pipeline stages small: parse, validate, filter, transform, aggregate.
- Remember that generator objects are one-shot iterators.
- Use a factory or reusable iterable class when repeated iteration is required.
- Prefer `yield from` for clean delegation.
- Capture `StopIteration.value` only intentionally; most code should use `for` loops.
- Validate arguments before yielding data.
- Make errors structured when processing long streams.